# Course notebook — Machine Learning Systems

Learn how models are trained, validated, diagnosed, and integrated into reliable software systems.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Problem framing and supervised models


### Lecture 01 · Problems, Data, and Baselines

Frame an ML task before choosing an algorithm. Define the unit of prediction, target, decision, population, time boundary, data availability, and a simple baseline.

**Predict before running.** Most model failures begin as problem or data-definition failures. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create a dataset split that respects time and entity boundaries.


In [ ]:
"""Lab 2.1: Problems, Data, and Baselines"""

apps = [
    {"id": 1, "income": 80000, "decision": 1},
    {"id": 2, "income": 40000, "decision": 0},
    {"id": 3, "income": 120000, "decision": 1},
]
baseline_rate = sum(a["decision"] for a in apps) / len(apps)
pred = 1 if baseline_rate >= 0.5 else 0
acc = sum(pred == a["decision"] for a in apps) / len(apps)
print({"majority_pred": pred, "accuracy": round(acc, 3)})


### Lecture 02 · Supervised Learning

Understand regression and classification as function approximation under a chosen loss. Connect linear models, trees, and neural networks through their assumptions.

**Predict before running.** The best model is the simplest one that meets the real decision requirement. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement linear and logistic regression before using a library.


In [ ]:
"""Lab 2.2: Supervised Learning"""

X = [[1.0, 0.5], [1.0, 1.2], [1.0, 2.0]]
y = [0, 0, 1]
w = [0.0, 0.0]
lr = 0.3
for _ in range(30):
    for xi, yi in zip(X, y):
        z = sum(wj*xij for wj, xij in zip(w, xi))
        pred = 1 / (1 + pow(2.718281828, -z))
        err = pred - yi
        w = [wj - lr * err * xij for wj, xij in zip(w, xi)]
print("weights:", [round(v, 3) for v in w])


**Section checkpoint.** Define a prediction cutoff and a naive baseline. Identify a feature available only after the cutoff.


## Representations and neural networks


### Lecture 03 · Unsupervised and Representation Learning

Use unlabeled data to discover structure, compress observations, and learn reusable representations.

**Predict before running.** Structure found by an algorithm is a hypothesis to validate, not a fact. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Cluster a dataset, visualize it, and explain why clusters are not automatically meaningful categories.


In [ ]:
"""Lab 2.3: Unsupervised and Representation Learning"""

import math
docs = {"payroll": "salary direct deposit batch", "retail": "card swipe retail purchase"}
words = sorted(set(w for d in docs.values() for w in d.split()))
def vec(doc):
    counts = {w: doc.split().count(w) for w in words}
    df = {w: sum(w in d.split() for d in docs.values()) for w in words}
    n = len(docs)
    return {w: counts[w] * math.log(n / df[w]) for w in words}
va, vb = vec(docs["payroll"]), vec(docs["retail"])
dot = sum(va[w]*vb[w] for w in words)
print("payroll vs retail dot:", round(dot, 3))


### Lecture 04 · Neural Networks

Build the mental model of layers, activations, losses, backpropagation, initialization, normalization, and optimization.

**Predict before running.** Neural networks learn compositions of transformations; training adjusts those transformations to reduce loss. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Train a small network and inspect gradients and learning curves.


In [ ]:
"""Lab 2.4: Neural Networks"""

def relu(x):
    return max(0.0, x)
x, w1, b1, w2, b2 = 1.5, 0.8, -0.2, 1.2, 0.1
hidden = relu(x * w1 + b1)
y = hidden * w2 + b2
loss = (y - 1.0) ** 2
grad_w2 = 2 * (y - 1.0) * hidden
print({"hidden": round(hidden, 3), "y": round(y, 3), "grad_w2": round(grad_w2, 3)})


**Section checkpoint.** Compare fitting a line with learning an intermediate representation. Name an input for which a line would be inadequate.


## Evidence and lifecycle


### Lecture 05 · Evaluation and Error Analysis

Choose metrics from the decision context, estimate uncertainty, inspect slices, and turn mistakes into the next experiment.

**Predict before running.** An aggregate metric can hide the exact population where a system is unsafe or useless. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Write an error taxonomy and compare two models with confidence intervals.


In [ ]:
"""Lab 2.5: Evaluation and Error Analysis"""

confusion = {"TP": 40, "FP": 10, "FN": 8, "TN": 142}
def metrics(c):
    prec = c["TP"] / (c["TP"] + c["FP"] + 1e-9)
    rec = c["TP"] / (c["TP"] + c["FN"] + 1e-9)
    return round(prec, 3), round(rec, 3)
slice_b = {"TP": 5, "FP": 12, "FN": 6, "TN": 20}
print("overall:", metrics(confusion))
print("slice_b:", metrics(slice_b))


### Lecture 06 · The ML Lifecycle

Connect data, experiments, models, releases, monitoring, drift, retraining, and retirement into an accountable lifecycle.

**Predict before running.** A trained model is an artifact; value and risk emerge from its full operating system. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Write a release checklist and a rollback plan for a prediction service.


In [ ]:
"""Lab 2.6: The ML Lifecycle"""

CHAPTER = "2.6"
print("chapter hook:", CHAPTER)
registry = {"model_v3": {"features": ["income", "debt"]}}
live = {"income": None, "debt": 1200}
def validate(row, schema):
    return [f for f in schema["features"] if row.get(f) is None]
issues = validate(live, registry["model_v3"])
print({"issues": issues, "action": "rollback" if issues else "serve"})
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Define a holdout slice and a rollback trigger. Explain which metric would detect the failure first.


## Course workshop

Add a held-out point and compare the trained line with the mean of the training labels. Keep the held-out point outside DATA.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Fit y = wx + b with batch gradient descent."""
DATA = [(0, 1), (1, 3), (2, 5), (3, 7)]
w = b = 0.0
rate = 0.05

for step in range(501):
    dw = db = loss = 0.0
    for x, y in DATA:
        error = (w * x + b) - y
        loss += error * error
        dw += 2 * error * x / len(DATA)
        db += 2 * error / len(DATA)
    w -= rate * dw
    b -= rate * db
    if step % 100 == 0:
        print(f"step={step:3d} loss={loss/len(DATA):.6f} w={w:.3f} b={b:.3f}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
held_out = (4, 9)
x, expected = held_out
prediction = w * x + b
mean_label = sum(y for _, y in DATA) / len(DATA)
assert abs(prediction - expected) < abs(mean_label - expected)
print('Workshop checks passed')


## Course project

Deliver a prediction service with a data card, baseline, evaluation report, error analysis, and monitoring plan.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
